# Clean up resources

Deletes the resource group created by `infra/main.bicep`, removing the
Foundry account, project, model deployment, Bing.Grounding resource, and
project connection in a single shot.

**All notebooks share this infrastructure.** Running this notebook
tears down infra for every lab in the repo.

Prerequisites:
- `az login`
- `.env` populated with `AZURE_SUBSCRIPTION_ID` and `RESOURCE_GROUP`


In [ ]:
import os
import shutil
import subprocess

try:
    from dotenv import find_dotenv, load_dotenv
    _dotenv_path = find_dotenv(usecwd=True)
    if _dotenv_path:
        load_dotenv(_dotenv_path, override=False)
        print(f"dotenv: loaded from {_dotenv_path}")
except ImportError:
    print("python-dotenv not installed; relying on process environment.")

SUBSCRIPTION_ID = (os.environ.get("AZURE_SUBSCRIPTION_ID") or "").strip()
RESOURCE_GROUP = (os.environ.get("RESOURCE_GROUP") or "rg-foundry-bing-research").strip()
if not SUBSCRIPTION_ID:
    raise RuntimeError("AZURE_SUBSCRIPTION_ID is required in .env.")

_AZ = shutil.which("az")
if not _AZ:
    raise RuntimeError("Azure CLI not found on PATH.")

print(f"Subscription : {SUBSCRIPTION_ID}")
print(f"Resource grp : {RESOURCE_GROUP}")


In [ ]:
# Confirm the resource group exists before deleting.
show = subprocess.run(
    [_AZ, "group", "show", "-n", RESOURCE_GROUP, "--subscription", SUBSCRIPTION_ID, "--only-show-errors"],
    capture_output=True, text=True,
)
if show.returncode != 0:
    print(f"Resource group {RESOURCE_GROUP} does not exist — nothing to clean up.")
else:
    print(f"Deleting resource group {RESOURCE_GROUP} (this may take a few minutes)...")
    res = subprocess.run(
        [_AZ, "group", "delete", "-n", RESOURCE_GROUP, "--subscription", SUBSCRIPTION_ID, "--yes", "--only-show-errors"],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        raise RuntimeError(f"az group delete failed: {res.stderr.strip()}")
    print(f"Resource group {RESOURCE_GROUP} deleted.")
